In [ ]:
import torch
import torch.nn as nn

In [ ]:
class RNNPositionalEncoding(nn.Module):
    """
    Positional encoding using a GRU (Gated Recurrent Unit) to generate
    learned, dynamic position encodings, as described in papers relating
    transformers to hippocampal models.

    This module uses an RNN to process a sequence of zero vectors, and its
    hidden states are used as the positional encodings.
    """

    def __init__(
        self,
        max_length: int,
        embed_dim: int,
        hidden_dim: int = None,
        num_layers: int = 1,
    ):
        """
        Initializes the RNNPositionalEncoding module.

        Args:
            max_length (int): The maximum sequence length that this module
                              will be used for. Not used.
            embed_dim (int): The dimensionality of the input embeddings. The
                             positional encodings will also have this dimension.
            hidden_dim (int, optional): The dimensionality of the RNN's hidden
                                        state. If None, it defaults to embed_dim.
            num_layers (int, optional): The number of layers in the RNN.
                                        Defaults to 1.
        """
        super().__init__()
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim if hidden_dim is not None else embed_dim
        self.num_layers = num_layers

        # The core RNN (GRU) that learns to generate positional patterns.
        self.rnn = nn.GRU(
            input_size=embed_dim,
            hidden_size=self.hidden_dim,
            num_layers=num_layers,
            batch_first=True,  # Crucial for [B, L, D] input shape
        )
        # self.input_param = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        # If the RNN's hidden dimension is different from the embedding dimension,
        # use linear layer to project it back to the correct size.
        if self.hidden_dim != embed_dim:
            self.proj = nn.Linear(self.hidden_dim, embed_dim)
        else:
            # If dimensions match, no projection is needed.
            self.proj = nn.Identity()

    def _generate_encodings(self, batch_size: int, seq_len: int, device: torch.device):
        """
        Internal helper function to generate the positional encodings.
        """
        # Create a dummy input tensor of zeros.
        # Shape: [B, L, embed_dim]
        # Use a trainable parameter as the input for each position (shared across positions)
        dummy_input = torch.zeros(batch_size, seq_len, self.embed_dim, device=device)
        # dummy_input = self.input_param.expand(batch_size, seq_len, self.embed_dim)

        # The GRU returns the output (hidden states for each time step) and
        # the final hidden state. We only need the former.
        # output shape: [B, L, hidden_dim]
        pos_enc, _ = self.rnn(dummy_input)

        # Project the encodings to the correct embedding dimension if necessary.
        # projected_pos_enc shape: [B, L, embed_dim]
        projected_pos_enc = self.proj(pos_enc)

        return projected_pos_enc

    def forward(self, feat: torch.Tensor):
        """
        Adds positional encoding to a complete input sequence.

        Args:
            feat: Input tensor of shape [B, L, D], where B is the batch size,
                  L is the sequence length, and D is the embedding dimension.

        Returns:
            A tensor of shape [B, L, D] with positional encodings added.
        """
        B, L, = feat.shape[0], feat.shape[1]
        # Generate positional encodings for the entire sequence length.
        pos_enc = self._generate_encodings(B, L, feat.device)

        return pos_enc

    def forward_with_position(self, feat: torch.Tensor, position: int):
        """
        Adds positional encoding at a specific position. This is useful for
        autoregressive decoding where inputs are processed one at a time.

        Args:
            feat: Input tensor of shape [B, 1, D] for the single item.
            position: The position index (integer) to generate the encoding for.

        Returns:
            A tensor of shape [B, 1, D] with the specific positional encoding added.
        """
        B = feat.shape[0]
        # Generate encodings for all positions up to and including `position`.
        # We need to run the RNN sequentially to get the correct hidden state.
        # Shape: [B, position + 1, D]
        all_pos_enc = self._generate_encodings(B, position + 1, feat.device)

        # Select the encoding for the specific position we need.
        # The shape becomes [B, 1, D] to match the input `feat`.
        pos_enc_at_position = all_pos_enc[:, position:position+1, :]
        
        return  pos_enc_at_position



In [ ]:
# class RNNPositionalEncoding(nn.Module):
# """
# Positional encoding using RNN (GRU) to generate position encodings.
# """

# def __init__(
#     self,
#     max_length: int,
#     embed_dim: int,
#     hidden_dim: int = None,
#     num_layers: int = 1,
# ):
#     super().__init__()
#     self.max_length = max_length
#     self.embed_dim = embed_dim
#     self.hidden_dim = hidden_dim if hidden_dim is not None else embed_dim
#     self.num_layers = num_layers

#     self.rnn = nn.GRU(
#         input_size=embed_dim,
#         hidden_size=self.hidden_dim,
#         num_layers=num_layers,
#         batch_first=True,
#     )

#     if self.hidden_dim != embed_dim:
#         self.proj = nn.Linear(self.hidden_dim, embed_dim)
#     else:
#         self.proj = nn.Identity()

#     # Position embeddings as input to RNN
#     self.pre_embeddings = nn.Embedding(max_length, embed_dim)

# def get_position_encodings(
#     self, batch_size: int, seq_len: int, device: torch.device
# ):
#     """Generate position encodings for given batch size and sequence length"""
#     # Create position indices [0, 1, ..., seq_len-1]
#     positions = (
#         torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)
#     )  # [B, L]
#     # Get position embeddings [B, L, D]
#     pre_emb = self.pre_embeddings(positions)
#     # Process through GRU
#     pos_encoding, _ = self.rnn(pre_emb)  # [B, L, hidden_dim]

#     return pos_encoding

# def forward(self, feat):
#     """Add positional encoding to the input features
#     Args:
#         feat: Input tensor of shape [B, L, D]
#     Returns:
#         Tensor of shape [B, L, D] with added positional encodings
#     """
#     B, L, _ = feat.shape
#     pos_enc = self.get_position_encodings(B, L, feat.device)  # [B, L, D]
#     if self.hidden_dim != self.embed_dim:
#         # Project to original dimension if needed
#         pos_enc = self.proj(pos_enc)  # [B, L, D]
#     return pos_enc

# def forward_with_position(self, feat, position: int):
#     """Add positional encoding at a specific position
#     Args:
#         feat: Input tensor of shape [B, 1, D]
#         position: Position index to add encoding for
#     Returns:
#         Tensor of shape [B, 1, D] with added positional encoding
#     """

#     B, L, _ = feat.shape
#     assert L == 1, "Input feature should have length 1 at dim 1"

#     # Run the RNN with the full sequence up to this point
#     rnn_output = self.get_position_encodings(
#         B, position + 1, feat.device
#     )  # [B, L, D]
#     # Get only the last position's output
#     pos_enc = rnn_output[:, -1:, :]  # [B, 1, hidden_dim]
#     if self.hidden_dim != self.embed_dim:
#         # Project to original dimension if needed
#         pos_enc = self.proj(pos_enc)  # [B, 1, D]
#     return pos_enc

In [ ]:
# Example Usage
if __name__ == '__main__':
    # --- Configuration ---
    max_len = 50
    emb_dim = 16
    hid_dim = 16  # Using a different hidden dim to test projection
    batch_size = 3
    seq_length = 8 # Example sequence length

    # --- Model Initialization ---
    rnn_pos_encoder = RNNPositionalEncoding(
        max_length=max_len,
        embed_dim=emb_dim,
        hidden_dim=hid_dim,
        num_layers=2
    )
    print("Model initialized:")
    print(rnn_pos_encoder)

    # --- Test `forward()` method ---
    print("\n--- Testing forward() method ---")
    # Create a dummy feature tensor
    input_features = torch.randn(batch_size, seq_length, emb_dim)
    print(f"Input feature shape: {input_features.shape}")

    # Get the output with positional encodings
    output_features = rnn_pos_encoder(input_features)
    print(f"Output feature shape: {output_features.shape}")
    
    # Check that the shapes are correct
    assert input_features.shape == output_features.shape

    # --- Test `forward_with_position()` method ---
    print("\n--- Testing forward_with_position() method ---")
    # Create a dummy feature tensor for a single item
    single_feature = torch.randn(batch_size, 1, emb_dim)
    target_position = 15 # e.g., we are at the 16th token
    print(f"Input single feature shape: {single_feature.shape}")
    print(f"Target position: {target_position}")
    
    # Get the output with positional encoding for that specific position
    output_single_feature = rnn_pos_encoder.forward_with_position(single_feature, target_position)
    print(f"Output single feature shape: {output_single_feature.shape}")

    # Check that the shapes are correct
    assert single_feature.shape == output_single_feature.shape
    
    print("\nAll tests passed successfully!")
